# 08 - SHAP Explainability Analysis

Use SHAP (SHapley Additive exPlanations) to interpret model predictions.

**Analyses**:
1. Beeswarm plot for combined 17-feature Gradient Boosting model
2. Beeswarm plot for pathway-only 8-feature Gradient Boosting model
3. Waterfall plot for highest-risk patient
4. Waterfall plot for lowest-risk patient

**Requires**: Downloaded GSE96058 expression data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from src.data_loader import load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, encode_clinical_features, filter_outcome
from src.features import compute_pathway_scores, add_ratio_features, build_feature_matrix
from src.models import get_classifiers
from sklearn.preprocessing import StandardScaler

## 1. Prepare Data

In [ ]:
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
gse_clin = filter_outcome(gse_clin)
gse_exp = load_gse96058_expression('../data/raw/GSE96058_gene_expression.csv')

common = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
gse_clin = gse_clin[gse_clin['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
gse_exp = gse_exp[gse_exp['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)

gse_exp_norm = zscore_normalize(gse_exp)
pathway_features = compute_pathway_scores(gse_exp_norm)
pathway_features = add_ratio_features(pathway_features)
clinical_features = encode_clinical_features(gse_clin)
combined_features = build_feature_matrix(pathway_features, clinical_features)
y = gse_clin['high_risk'].values

print(f"Combined features: {combined_features.shape}")
print(f"Pathway features: {pathway_features.shape}")

## 2. Train Models on Full Dataset

In [ ]:
# Combined model (17 features)
scaler_combined = StandardScaler()
X_combined_s = scaler_combined.fit_transform(combined_features)
X_combined_df = pd.DataFrame(X_combined_s, columns=combined_features.columns)

gb_combined = get_classifiers()['Gradient Boosting']
gb_combined.fit(X_combined_s, y)
print("Trained GB on combined features (17)")

# Pathway-only model (8 features)
scaler_pathway = StandardScaler()
X_pathway_s = scaler_pathway.fit_transform(pathway_features)
X_pathway_df = pd.DataFrame(X_pathway_s, columns=pathway_features.columns)

gb_pathway = get_classifiers()['Gradient Boosting']
gb_pathway.fit(X_pathway_s, y)
print("Trained GB on pathway features (8)")

## 3. SHAP Beeswarm - Combined Model

In [ ]:
print("Computing SHAP values for combined model...")
explainer_combined = shap.TreeExplainer(gb_combined)
shap_values_combined = explainer_combined.shap_values(X_combined_df)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_combined, X_combined_df, show=False)
plt.title('SHAP Feature Importance - Combined Model (17 Features)', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/fig_shap_beeswarm.png', dpi=300, bbox_inches='tight')
print("Saved figures/fig_shap_beeswarm.png")
plt.show()

## 4. SHAP Beeswarm - Pathway-Only Model

In [ ]:
print("Computing SHAP values for pathway-only model...")
explainer_pathway = shap.TreeExplainer(gb_pathway)
shap_values_pathway = explainer_pathway.shap_values(X_pathway_df)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_pathway, X_pathway_df, show=False)
plt.title('SHAP Feature Importance - Pathway-Only Model (8 Features)', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/fig_shap_pathway_only.png', dpi=300, bbox_inches='tight')
print("Saved figures/fig_shap_pathway_only.png")
plt.show()

## 5. Waterfall Plots

In [ ]:
# Find highest and lowest risk patients
fx_values = shap_values_combined.sum(axis=1) + explainer_combined.expected_value
high_idx = np.argmax(fx_values)
low_idx = np.argmin(fx_values)

print(f"Highest risk patient: index {high_idx}, f(x) = {fx_values[high_idx]:.4f}")
print(f"Lowest risk patient:  index {low_idx}, f(x) = {fx_values[low_idx]:.4f}")

In [ ]:
# Waterfall - highest risk
shap_explanation = shap.Explanation(
    values=shap_values_combined[high_idx],
    base_values=explainer_combined.expected_value,
    data=X_combined_df.iloc[high_idx].values,
    feature_names=combined_features.columns.tolist()
)

plt.figure(figsize=(10, 8))
shap.plots.waterfall(shap_explanation, show=False)
plt.title('SHAP Waterfall - Highest Risk Patient', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/fig_waterfall_high.png', dpi=300, bbox_inches='tight')
print("Saved figures/fig_waterfall_high.png")
plt.show()

In [ ]:
# Waterfall - lowest risk
shap_explanation_low = shap.Explanation(
    values=shap_values_combined[low_idx],
    base_values=explainer_combined.expected_value,
    data=X_combined_df.iloc[low_idx].values,
    feature_names=combined_features.columns.tolist()
)

plt.figure(figsize=(10, 8))
shap.plots.waterfall(shap_explanation_low, show=False)
plt.title('SHAP Waterfall - Lowest Risk Patient', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/fig_waterfall_low.png', dpi=300, bbox_inches='tight')
print("Saved figures/fig_waterfall_low.png")
plt.show()

## 6. Save SHAP Values

In [ ]:
shap_df = pd.DataFrame(shap_values_combined, columns=combined_features.columns)
shap_df.to_csv('../results/04_shap_values_all_patients.csv', index=False)
print(f"Saved SHAP values for all {len(shap_df)} patients to results/04_shap_values_all_patients.csv")